# Local Network Test

This notebook loads the saved 32-neuron heterogeneous SHD checkpoint as **Local-Het**, builds a matching 32-neuron homogeneous control as **Local-Hom**, and reuses the post-training evaluation flow from the initial comparison notebook.

Planned flow:
1. Load the `network_A_checkpoint.pt` model as Local-Het.
2. Build and train Local-Hom with the same architecture and optimizer settings.
3. Run downstream evaluation sections: task performance, tau analysis, and a primary zero-$M$ null-distribution z-score analysis across subset sizes.
4. Keep a second, more stringent matched-information null implementation as a deferred design section for later activation.


In [1]:
from __future__ import annotations

import io
import sys
import json
import random
import time
from contextlib import contextmanager, redirect_stdout
from copy import deepcopy
from pathlib import Path

import importlib
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm, rankdata

import torch

PROJECT_ROOT = Path(r"C:\Users\Priya\Desktop\research project (SNN Info Theory)")
WIMFO_ROOT = PROJECT_ROOT / "wimfo"
PAPER_ROOT = PROJECT_ROOT / "neural_heterogeneity" / "SuGD_code"
for extra_path in [WIMFO_ROOT, PAPER_ROOT]:
    if str(extra_path) not in sys.path:
        sys.path.insert(0, str(extra_path))

from clipper import ZeroOneClipper
from wimfo.W_M_Info import W_M_calculator
from wimfo.utils.utils_gauss import get_cov
from data_gen import open_file, sparse_data_generator
from reg_loss import loss as repo_loss
from SuSpike import SuSpike

RSNN = importlib.import_module("model").RSNN
clipper = ZeroOneClipper()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
spike_fn = SuSpike.apply

SHD_TRAIN = PROJECT_ROOT / "data" / "shd" / "shd_train.h5"
SHD_TEST = PROJECT_ROOT / "data" / "shd" / "shd_test.h5"
CHECKPOINT_PATH = PROJECT_ROOT / "Project Files" / "network_A_checkpoint.pt"

print(f"Device: {DEVICE}")
print(f"Train file exists: {SHD_TRAIN.exists()}")
print(f"Test file exists: {SHD_TEST.exists()}")
print(f"Checkpoint exists: {CHECKPOINT_PATH.exists()}")


Device: cuda
Train file exists: True
Test file exists: True
Checkpoint exists: True


In [5]:
# Core helpers copied from the working Architecture Test notebook.

SHD_DEFAULTS = {
    "seed": 1000,
    "dtype": torch.float,
    "device": DEVICE,
    "cuda": DEVICE.type == "cuda",
    "nb_inputs": 700,
    "nb_hidden": [],
    "nb_recurrent": 32,
    "nb_outputs": 20,
    "batch_size": 256,
    "time_step": 1.0e-3,
    "nb_steps": 1000,
    "tau_syn": 10e-3,
    "tau_mem": 20e-3,
    "threshold": 1.0,
    "tref": 0.0,
    "dist": "gamma",
    "dist_prms": 3.0,
    "lr": 4e-3,
    "lr_ab": 4e-3,
    "betas": (0.9, 0.999),
    "weight_decay": 0.0,
    "nb_epochs": 25,
    "drop_last": True,
    "sl": 0.0,
    "thetal": 0.0,
    "su": 0.0,
    "thetau": 0.0,
    "rate": 0.0,
    "p_del": 0.0,
    "train_th": 0,
    "het_th": 0,
    "train_reset": 0,
    "het_reset": 0,
    "train_rest": 0,
    "het_rest": 0,
    "sparse_data_generator": "sparse_data_generator",
    "time_scale": [0.5, 1.0],
    "model": "RSNN",
    "savestep": 10,
    "clip": 1,
    "plot_step": 50,
    "class_list": list(range(20)),
    "het_ab": 1,
    "train_ab": 1,
    "train_hom_ab": 0,
}

RUN_CONFIG = {
    "m_subset_size": 6,
    "m_batches": 3,
    "m_downsample_stride": 4,
    "m_ridge": 1e-2,
    "null_samples": 200,
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def decay_to_tau(decay, dt):
    decay = np.asarray(decay, dtype=np.float64)
    return -dt / np.log(decay)


@contextmanager
def shd_open(path):
    units, times, labels = open_file(str(path))
    try:
        yield units, times, labels
    finally:
        units._v_file.close()


class SHDCache:
    def __init__(self, path):
        raw_u, raw_t, raw_l = open_file(str(path))
        self.units = list(raw_u[:])
        self.times = list(raw_t[:])
        self.labels = np.array(raw_l[:])
        raw_u._v_file.close()
        print(f"  SHDCache: {len(self.labels)} samples loaded from {Path(str(path)).name}")

    def __len__(self):
        return len(self.labels)


def _is_cache(obj):
    return hasattr(obj, "units") and hasattr(obj, "times") and hasattr(obj, "labels")


@contextmanager
def shd_open_cached(cache):
    yield cache.units, cache.times, cache.labels


def fast_sparse_data_generator(units, times, labels, prms, shuffle=True, epoch=0, drop_last=True):
    rate = prms.get("rate", 0.0)
    p_del = prms.get("p_del", 0.0)
    if rate != 0.0 or p_del != 0.0:
        yield from sparse_data_generator(units, times, labels, prms, shuffle=shuffle, epoch=epoch, drop_last=drop_last)
        return

    seed = prms["seed"] + epoch
    batch_size = prms["batch_size"]
    nb_steps = prms["nb_steps"]
    nb_units = prms["nb_inputs"]
    inv_dt = 1.0 / prms["time_step"]
    class_list = prms["class_list"]

    label_arr = labels if isinstance(labels, np.ndarray) else np.array(labels[:])
    sample_index = np.where(np.isin(label_arr, class_list))[0]
    num_samples = len(sample_index)
    n_batches = (num_samples // batch_size) if drop_last else -(-num_samples // batch_size)

    np.random.seed(seed)
    if shuffle:
        np.random.shuffle(sample_index)

    for counter in range(n_batches):
        batch_index = sample_index[batch_size * counter:min(num_samples, batch_size * (counter + 1))]
        actual_bs = len(batch_index)

        t_arrays = [np.round(times[idx] * inv_dt).astype(np.int64) for idx in batch_index]
        u_arrays = [units[idx] for idx in batch_index]
        lengths = np.array([len(a) for a in t_arrays], dtype=np.int64)

        if lengths.sum():
            all_ts = np.concatenate(t_arrays)
            all_us = np.concatenate(u_arrays)
            all_bc = np.repeat(np.arange(actual_bs, dtype=np.int64), lengths)
            valid = all_ts < nb_steps
            all_ts, all_us, all_bc = all_ts[valid], all_us[valid], all_bc[valid]
            i = torch.from_numpy(np.stack([all_bc, all_ts, all_us]))
            v = torch.ones(all_ts.size, dtype=torch.float32)
            X_batch = torch.sparse_coo_tensor(i, v, torch.Size([actual_bs, nb_steps, nb_units])).to_dense()
        else:
            X_batch = torch.zeros(actual_bs, nb_steps, nb_units)

        X_batch.clamp_(max=1.0)
        y_batch = torch.tensor([class_list.index(int(a)) for a in label_arr[batch_index]], dtype=torch.long)
        yield X_batch, y_batch


def shd_generator(units, times, labels, prms, shuffle, epoch, drop_last):
    yield from fast_sparse_data_generator(units, times, labels, prms, shuffle=shuffle, epoch=epoch, drop_last=drop_last)


def forward_logits(model, x):
    layer_recs = model(0, 0, x)
    output_layer = layer_recs[-1]
    logits, _ = torch.max(output_layer[1], dim=1)
    return logits, layer_recs


def get_hidden_ab_tensors(model):
    hidden_layer = model.network[0]
    return hidden_layer.alpha.detach().clone(), hidden_layer.beta.detach().clone()


def summarize_hidden_taus(model, time_step: float):
    hidden_layer = model.network[0]
    alpha = hidden_layer.alpha.detach().cpu().numpy().ravel()
    beta = hidden_layer.beta.detach().cpu().numpy().ravel()
    return {
        "alpha_unique": int(np.unique(np.round(alpha, 8)).size),
        "beta_unique": int(np.unique(np.round(beta, 8)).size),
        "tau_syn_ms_min": float(decay_to_tau(alpha, time_step).min() * 1e3),
        "tau_syn_ms_max": float(decay_to_tau(alpha, time_step).max() * 1e3),
        "tau_mem_ms_min": float(decay_to_tau(beta, time_step).min() * 1e3),
        "tau_mem_ms_max": float(decay_to_tau(beta, time_step).max() * 1e3),
    }


def count_epoch_samples(sample_count, batch_size, drop_last, batch_limit=None):
    if drop_last:
        available = sample_count // batch_size
    else:
        available = -(-sample_count // batch_size)
    used = available if batch_limit is None else min(available, int(batch_limit))
    if drop_last:
        return used * batch_size
    remaining, total = sample_count, 0
    for _ in range(used):
        take = min(batch_size, remaining)
        total += take
        remaining -= take
    return total


def make_optimizer(model, prms):
    weight_params, ab_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "alpha" in name or "beta" in name:
            ab_params.append(param)
        else:
            weight_params.append(param)
    param_groups = [{"params": weight_params, "lr": prms["lr"], "weight_decay": prms["weight_decay"]}]
    if ab_params:
        param_groups.append({"params": ab_params, "lr": prms["lr_ab"]})
    return torch.optim.Adam(param_groups, betas=prms["betas"])


@torch.no_grad()
def evaluate_batches(model, prms, units, times, labels, max_batches=None, num_samples=None, use_amp=True):
    if num_samples is None:
        total = int(np.isin(labels[:], prms["class_list"]).sum())
        num_samples = count_epoch_samples(total, prms["batch_size"], drop_last=False, batch_limit=max_batches)
    use_amp = use_amp and DEVICE.type == "cuda"
    model.eval()
    loss_acc, correct = 0.0, 0
    for idx, (x, y) in enumerate(shd_generator(units, times, labels, prms, shuffle=False, epoch=0, drop_last=False)):
        if max_batches is not None and idx >= max_batches:
            break
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
            logits, layer_recs = forward_logits(model, x)
            loss_acc += repo_loss(logits, layer_recs, y, num_samples, prms).item()
        correct += (logits.argmax(1) == y).sum().item()
    return {"loss": loss_acc, "acc": correct / max(num_samples, 1), "n": num_samples}


def train_experiment(model, prms, train_data, test_data, use_amp=True):
    use_amp = use_amp and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    optimizer = make_optimizer(model, prms)
    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    tr_ctx = shd_open_cached if _is_cache(train_data) else shd_open
    te_ctx = shd_open_cached if _is_cache(test_data) else shd_open
    with tr_ctx(train_data) as (u_tr, t_tr, l_tr), te_ctx(test_data) as (u_te, t_te, l_te):
        if not prms["class_list"]:
            prms["class_list"] = np.unique(l_tr[:]).tolist()

        total_tr = int(np.isin(l_tr[:], prms["class_list"]).sum())
        total_te = int(np.isin(l_te[:], prms["class_list"]).sum())
        eff_tr = count_epoch_samples(total_tr, prms["batch_size"], drop_last=bool(prms["drop_last"]))
        eff_te = count_epoch_samples(total_te, prms["batch_size"], drop_last=False)

        if prms["clip"]:
            model.apply(clipper)

        for epoch in range(1, prms["nb_epochs"] + 1):
            ep_t0 = time.perf_counter()
            model.train()
            ep_loss, ep_correct = 0.0, 0
            for x, y in shd_generator(u_tr, t_tr, l_tr, prms, shuffle=True, epoch=epoch, drop_last=prms["drop_last"]):
                x, y = x.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                    logits, layer_recs = forward_logits(model, x)
                    loss_val = repo_loss(logits, layer_recs, y, eff_tr, prms)
                scaler.scale(loss_val).backward()
                scaler.step(optimizer)
                scaler.update()
                if prms["clip"]:
                    model.apply(clipper)
                ep_loss += loss_val.item()
                ep_correct += (logits.argmax(1) == y).sum().item()

            test_m = evaluate_batches(model, prms, u_te, t_te, l_te, num_samples=eff_te, use_amp=use_amp)
            history["train_loss"].append(ep_loss)
            history["train_acc"].append(ep_correct / max(eff_tr, 1))
            history["test_loss"].append(test_m["loss"])
            history["test_acc"].append(test_m["acc"])
            ep_elapsed = time.perf_counter() - ep_t0
            print(f"epoch={epoch:03d} train_acc={ep_correct/max(eff_tr,1):.3f} test_acc={test_m['acc']:.3f} ({ep_elapsed/60:.1f} min)")

    return history


def select_hidden_subset(nb_hidden: int, subset_size: int = 8) -> np.ndarray:
    return np.linspace(0, nb_hidden - 1, min(subset_size, nb_hidden), dtype=int)


def gaussian_copula_normalize(data: np.ndarray) -> np.ndarray:
    transformed = np.zeros_like(data, dtype=np.float64)
    for index, row in enumerate(data):
        if np.allclose(row, row[0]):
            continue
        ranks = rankdata(row, method="average")
        uniform = np.clip((ranks - 0.5) / len(row), 1e-6, 1.0 - 1e-6)
        transformed[index] = norm.ppf(uniform)
    return transformed


def regularize_covariance(cov: np.ndarray, ridge: float = 1e-2) -> np.ndarray:
    cov = np.asarray(cov, dtype=np.float64)
    cov = 0.5 * (cov + cov.T)
    scale = np.trace(cov) / max(cov.shape[0], 1)
    if not np.isfinite(scale) or scale <= 0.0:
        scale = 1.0
    return cov + ridge * scale * np.eye(cov.shape[0], dtype=np.float64)


@torch.no_grad()
def collect_hidden_state_matrix(model, prms, path, max_batches=2, subset_size=6, downsample_stride=4, subset_indices=None):
    with shd_open(path) as (units, times, labels):
        hidden_size = int(model.network[0].output_size)
        if subset_indices is None:
            chosen = select_hidden_subset(hidden_size, subset_size=subset_size)
        else:
            chosen = np.asarray(subset_indices, dtype=int)
        stride = max(int(downsample_stride), 1)
        chunks = []
        for batch_index, (x, _) in enumerate(shd_generator(units, times, labels, prms, shuffle=False, epoch=0, drop_last=False)):
            if batch_index >= max_batches:
                break
            x = x.to(DEVICE)
            layer_recs = model(0, 0, x)
            hidden_mem = layer_recs[0][1][:, ::stride, :][:, :, chosen].detach().cpu().numpy()
            hidden_mem = np.transpose(hidden_mem, (2, 0, 1)).reshape(len(chosen), -1)
            chunks.append(hidden_mem)
    if not chunks:
        raise RuntimeError("No SHD batches were collected for M-information estimation.")
    return np.concatenate(chunks, axis=1)


def compute_w_m_from_hidden_matrix(hidden_data, lag=1, ridge=1e-2, optimiser_order=(("Newton", None), ("Adam", {"atol": 1e-4, "rtol": 1e-4}))):
    gaussian_data = gaussian_copula_normalize(hidden_data)
    lagged_cov = regularize_covariance(get_cov(gaussian_data, t=lag), ridge=ridge)
    last_error = None
    for optimiser, options in optimiser_order:
        try:
            with io.StringIO() as buffer, redirect_stdout(buffer):
                W_bits, M_bits = W_M_calculator(
                    lagged_cov,
                    option="distr",
                    type="gaussian",
                    unit="bits",
                    verbose=False,
                    optimiser=optimiser,
                    options=options,
                )
        except Exception as exc:
            last_error = exc
            continue
        if np.isfinite(W_bits) and np.isfinite(M_bits):
            return {
                "W_bits": float(W_bits),
                "M_bits": float(M_bits),
                "samples": int(gaussian_data.shape[1]),
                "optimiser": optimiser,
                "lagged_cov": lagged_cov,
            }
    if last_error is not None:
        raise RuntimeError("W/M calculation failed.") from last_error
    raise RuntimeError("W/M calculation returned NaN for all tested configurations.")


def estimate_m_info(model, prms, path, subset_size=6, max_batches=2, lag=1, downsample_stride=4, ridge=1e-2, exact_subset_size=False):
    if exact_subset_size:
        candidate_sizes = [int(subset_size)]
        candidate_strides = [max(int(downsample_stride), 1)]
    else:
        candidate_sizes = [v for v in [subset_size, 4, 3, 2] if v > 0]
        candidate_strides = list(dict.fromkeys([max(int(v), 1) for v in [downsample_stride, downsample_stride * 2, 1]]))

    last_error = None
    for candidate_size in candidate_sizes:
        for candidate_stride in candidate_strides:
            hidden_data = collect_hidden_state_matrix(
                model,
                prms,
                path,
                max_batches=max_batches,
                subset_size=candidate_size,
                downsample_stride=candidate_stride,
            )
            try:
                result = compute_w_m_from_hidden_matrix(hidden_data, lag=lag, ridge=ridge)
            except Exception as exc:
                last_error = exc
                continue
            result.update({
                "subset_size": int(candidate_size),
                "downsample_stride": int(candidate_stride),
                "ridge": float(ridge),
            })
            return result
    if last_error is not None:
        raise RuntimeError("M-information estimation failed.") from last_error
    raise RuntimeError("M-information estimation returned NaN for all tested configurations.")


def simulate_zero_m_gaussian_hidden(nvar, nsamples, rng, mode="iid"):
    if mode == "iid":
        return rng.standard_normal((nvar, nsamples))
    if mode == "independent_ar1":
        coeffs = rng.uniform(-0.8, 0.8, size=nvar)
        noise = rng.standard_normal((nvar, nsamples))
        data = np.zeros((nvar, nsamples), dtype=np.float64)
        data[:, 0] = noise[:, 0]
        for index in range(1, nsamples):
            data[:, index] = coeffs * data[:, index - 1] + noise[:, index]
        return data
    raise ValueError(f"Unknown null mode: {mode}")


def zscore_from_null(observed, null_values):
    null_values = np.asarray(null_values, dtype=np.float64)
    valid = null_values[np.isfinite(null_values)]
    if valid.size == 0:
        return float("nan"), float("nan"), float("nan")
    mu = float(valid.mean())
    sigma = float(valid.std(ddof=1)) if valid.size > 1 else 0.0
    if sigma <= 0.0 or not np.isfinite(sigma):
        z = float("nan")
    else:
        z = float((observed - mu) / sigma)
    p_upper = float((1.0 + np.sum(valid >= observed)) / (valid.size + 1.0))
    return z, mu, p_upper


def zero_m_null_for_subset(model, prms, path, subset_size, max_batches=2, lag=1, downsample_stride=4, ridge=1e-2, n_null=200, null_mode="iid", seed=12345):
    hidden_data = collect_hidden_state_matrix(
        model,
        prms,
        path,
        max_batches=max_batches,
        subset_size=subset_size,
        downsample_stride=downsample_stride,
    )
    observed = compute_w_m_from_hidden_matrix(hidden_data, lag=lag, ridge=ridge)

    rng = np.random.default_rng(seed + int(subset_size))
    nvar, nsamples = hidden_data.shape
    null_w = []
    null_m = []
    errors = 0
    for _ in range(int(n_null)):
        null_hidden = simulate_zero_m_gaussian_hidden(nvar, nsamples, rng, mode=null_mode)
        try:
            null_result = compute_w_m_from_hidden_matrix(null_hidden, lag=lag, ridge=ridge)
            null_w.append(null_result["W_bits"])
            null_m.append(null_result["M_bits"])
        except Exception:
            null_w.append(float("nan"))
            null_m.append(float("nan"))
            errors += 1

    m_z, m_null_mean, m_p_upper = zscore_from_null(observed["M_bits"], null_m)
    w_z, w_null_mean, w_p_upper = zscore_from_null(observed["W_bits"], null_w)
    valid_m = int(np.isfinite(np.asarray(null_m, dtype=np.float64)).sum())
    valid_w = int(np.isfinite(np.asarray(null_w, dtype=np.float64)).sum())

    return {
        "subset_size": int(subset_size),
        "samples": int(observed["samples"]),
        "lag": int(lag),
        "downsample_stride": int(downsample_stride),
        "ridge": float(ridge),
        "null_mode": str(null_mode),
        "n_null": int(n_null),
        "n_null_errors": int(errors),
        "observed_W_bits": float(observed["W_bits"]),
        "observed_M_bits": float(observed["M_bits"]),
        "null_W_mean": float(w_null_mean),
        "null_M_mean": float(m_null_mean),
        "W_zscore": float(w_z),
        "M_zscore": float(m_z),
        "W_p_upper": float(w_p_upper),
        "M_p_upper": float(m_p_upper),
        "null_W_values": [float(v) if np.isfinite(v) else float("nan") for v in null_w],
        "null_M_values": [float(v) if np.isfinite(v) else float("nan") for v in null_m],
        "n_null_valid_W": valid_w,
        "n_null_valid_M": valid_m,
        "status": "ok" if valid_m > 0 and valid_w > 0 else "error",
    }


def sweep_zero_m_null_by_subset(model, prms, path, subset_sizes=None, max_batches=2, lag=1, downsample_stride=4, ridge=1e-2, n_null=200, null_mode="iid", seed=12345):
    hidden_size = int(model.network[0].output_size)
    if subset_sizes is None:
        subset_sizes = list(range(2, hidden_size + 1))

    results = []
    for subset_size in subset_sizes:
        subset_size = int(subset_size)
        if subset_size < 2 or subset_size > hidden_size:
            continue
        try:
            result = zero_m_null_for_subset(
                model,
                prms,
                path,
                subset_size=subset_size,
                max_batches=max_batches,
                lag=lag,
                downsample_stride=downsample_stride,
                ridge=ridge,
                n_null=n_null,
                null_mode=null_mode,
                seed=seed,
            )
            print(
                f"subset={subset_size:02d}  obsM={result['observed_M_bits']:.4f}  "
                f"nullM={result['null_M_mean']:.4f}  Mz={result['M_zscore']:.2f}  "
                f"p={result['M_p_upper']:.4f}"
            )
        except Exception as exc:
            result = {
                "subset_size": subset_size,
                "observed_W_bits": float("nan"),
                "observed_M_bits": float("nan"),
                "null_W_mean": float("nan"),
                "null_M_mean": float("nan"),
                "W_zscore": float("nan"),
                "M_zscore": float("nan"),
                "W_p_upper": float("nan"),
                "M_p_upper": float("nan"),
                "null_W_values": [],
                "null_M_values": [],
                "n_null_valid_W": 0,
                "n_null_valid_M": 0,
                "n_null_errors": int(n_null),
                "status": "error",
                "error": str(exc),
            }
            print(f"subset={subset_size:02d}  ERROR  {exc}")
        results.append(result)
    return results


def largest_valid_sweep_result(results, key_prefix="observed"):
    metric_key = "M_bits" if key_prefix == "observed" else "M_zscore"
    if key_prefix == "observed":
        valid = [row for row in results if np.isfinite(row.get("observed_W_bits", np.nan)) and np.isfinite(row.get("observed_M_bits", np.nan))]
    else:
        valid = [row for row in results if np.isfinite(row.get("M_zscore", np.nan))]
    if not valid:
        raise RuntimeError("No valid M-information results were produced in the subset sweep.")
    valid.sort(key=lambda row: row["subset_size"])
    return valid[-1]


print("Helper functions loaded.")


Helper functions loaded.


In [7]:
# Autosave/resume helpers for Implementation A.

def _sanitize_for_json(value):
    if isinstance(value, dict):
        return {str(key): _sanitize_for_json(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [_sanitize_for_json(item) for item in value]
    if isinstance(value, np.ndarray):
        return [_sanitize_for_json(item) for item in value.tolist()]
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, Path):
        return str(value)
    return value


def save_model_checkpoint(path, model, history, elapsed_s, prms, extra=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "model_state_dict": model.state_dict(),
        "history": history,
        "elapsed_s": float(elapsed_s),
        "prms": {k: v for k, v in prms.items() if not isinstance(v, (torch.dtype, torch.device))},
    }
    if extra:
        payload.update(extra)
    torch.save(payload, path)
    return path


def save_sweep_results(results, save_path, metadata=None):
    payload = {
        "metadata": _sanitize_for_json(metadata or {}),
        "results": _sanitize_for_json(results),
    }
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = save_path.with_suffix(save_path.suffix + ".tmp")
    with open(tmp_path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
    tmp_path.replace(save_path)


def load_sweep_results(save_path):
    save_path = Path(save_path)
    if not save_path.exists():
        return []
    with open(save_path, "r", encoding="utf-8") as handle:
        payload = json.load(handle)
    return payload.get("results", [])


def _resume_index(results):
    return {int(row["subset_size"]): row for row in results if "subset_size" in row}


def sweep_zero_m_null_by_subset(model, prms, path, subset_sizes=None, max_batches=2, lag=1, downsample_stride=4, ridge=1e-2, n_null=200, null_mode="iid", seed=12345, save_path=None, resume=True, label=None):
    hidden_size = int(model.network[0].output_size)
    if subset_sizes is None:
        subset_sizes = list(range(2, hidden_size + 1))

    existing_results = load_sweep_results(save_path) if resume and save_path is not None else []
    cached_by_subset = _resume_index(existing_results)
    results = []

    metadata = {
        "label": label,
        "hidden_size": hidden_size,
        "max_batches": max_batches,
        "lag": lag,
        "downsample_stride": downsample_stride,
        "ridge": ridge,
        "n_null": n_null,
        "null_mode": null_mode,
        "seed": seed,
    }

    for subset_size in subset_sizes:
        subset_size = int(subset_size)
        if subset_size < 2 or subset_size > hidden_size:
            continue

        cached = cached_by_subset.get(subset_size)
        if cached is not None:
            results.append(cached)
            print(f"subset={subset_size:02d}  resumed  Mz={cached.get('M_zscore', float('nan')):.2f}  p={cached.get('M_p_upper', float('nan')):.4f}")
            continue

        try:
            result = zero_m_null_for_subset(
                model,
                prms,
                path,
                subset_size=subset_size,
                max_batches=max_batches,
                lag=lag,
                downsample_stride=downsample_stride,
                ridge=ridge,
                n_null=n_null,
                null_mode=null_mode,
                seed=seed,
            )
            print(
                f"subset={subset_size:02d}  obsM={result['observed_M_bits']:.4f}  "
                f"nullM={result['null_M_mean']:.4f}  Mz={result['M_zscore']:.2f}  "
                f"p={result['M_p_upper']:.4f}"
            )
        except Exception as exc:
            result = {
                "subset_size": subset_size,
                "observed_W_bits": float("nan"),
                "observed_M_bits": float("nan"),
                "null_W_mean": float("nan"),
                "null_M_mean": float("nan"),
                "W_zscore": float("nan"),
                "M_zscore": float("nan"),
                "W_p_upper": float("nan"),
                "M_p_upper": float("nan"),
                "null_W_values": [],
                "null_M_values": [],
                "n_null_valid_W": 0,
                "n_null_valid_M": 0,
                "n_null_errors": int(n_null),
                "status": "error",
                "error": str(exc),
            }
            print(f"subset={subset_size:02d}  ERROR  {exc}")

        results.append(result)
        if save_path is not None:
            save_sweep_results(results, save_path, metadata=metadata)
            print(f"  saved -> {Path(save_path).name}")

    return results


print("Autosave/resume helpers loaded.")

Autosave/resume helpers loaded.


In [3]:
print("Pre-loading SHD training data into RAM...")
SHD_TRAIN_CACHE = SHDCache(SHD_TRAIN)
print("Pre-loading SHD test data into RAM...")
SHD_TEST_CACHE = SHDCache(SHD_TEST)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
saved_prms = dict(checkpoint["prms"])

local_het_prms = dict(SHD_DEFAULTS)
local_het_prms.update(saved_prms)
local_het_prms.update({
    "dtype": torch.float,
    "device": DEVICE,
    "cuda": DEVICE.type == "cuda",
    "het_ab": 1,
    "train_ab": 1,
    "train_hom_ab": 0,
})
local_het_prms["alpha"] = float(np.exp(-local_het_prms["time_step"] / local_het_prms["tau_syn"]))
local_het_prms["beta"] = float(np.exp(-local_het_prms["time_step"] / local_het_prms["tau_mem"]))

local_hom_prms = dict(local_het_prms)
local_hom_prms.update({
    "het_ab": 0,
    "train_ab": 0,
    "train_hom_ab": 1,
})
local_hom_prms["alpha"] = float(np.exp(-local_hom_prms["time_step"] / local_hom_prms["tau_syn"]))
local_hom_prms["beta"] = float(np.exp(-local_hom_prms["time_step"] / local_hom_prms["tau_mem"]))

LOCAL_HET_LABEL = "Local-Het"
LOCAL_HOM_LABEL = "Local-Hom"

set_seed(local_het_prms["seed"])
local_het_model = RSNN(local_het_prms, rec=True).to(DEVICE)
local_het_model.load_state_dict(checkpoint["model_state_dict"])
local_het_history = checkpoint["history"]
local_het_elapsed = float(checkpoint["elapsed_s"])

set_seed(local_hom_prms["seed"])
local_hom_model = RSNN(local_hom_prms, rec=True).to(DEVICE)
local_hom_alpha_before, local_hom_beta_before = get_hidden_ab_tensors(local_hom_model)
local_hom_before = summarize_hidden_taus(local_hom_model, local_hom_prms["time_step"])

local_het_alpha_after, local_het_beta_after = get_hidden_ab_tensors(local_het_model)
local_het_after = summarize_hidden_taus(local_het_model, local_het_prms["time_step"])

training_results = {
    LOCAL_HET_LABEL: {
        "test_acc": local_het_history["test_acc"][-1],
        "initial_summary": None,
        "final_summary": local_het_after,
    }
}

print(f"Loaded {LOCAL_HET_LABEL} checkpoint with final test acc = {local_het_history['test_acc'][-1]:.3f}")
print(f"{LOCAL_HOM_LABEL} params: nb_recurrent={local_hom_prms['nb_recurrent']}  dt={local_hom_prms['time_step']*1e3:.1f}ms  steps={local_hom_prms['nb_steps']}  bs={local_hom_prms['batch_size']}  epochs={local_hom_prms['nb_epochs']}")


Pre-loading SHD training data into RAM...
  SHDCache: 8156 samples loaded from shd_train.h5
Pre-loading SHD test data into RAM...
  SHDCache: 2264 samples loaded from shd_test.h5
Loaded Local-Het checkpoint with final test acc = 0.649
Local-Hom params: nb_recurrent=32  dt=1.0ms  steps=1000  bs=256  epochs=25


## Train Local-Hom

`Local-Het` is loaded from the saved checkpoint. This cell trains the matched homogeneous control `Local-Hom` with the same 32-neuron architecture, 1 ms timestep, 1000 steps, batch size 256, and 25 epochs.


In [ ]:
# Train Local-Hom, compare to the saved Local-Het checkpoint, and persist the homogeneous model.
train_t0 = time.perf_counter()
local_hom_history = train_experiment(
    local_hom_model,
    local_hom_prms,
    SHD_TRAIN_CACHE,
    SHD_TEST_CACHE,
    use_amp=True,
)
local_hom_elapsed = time.perf_counter() - train_t0

local_hom_checkpoint_path = save_model_checkpoint(
    PROJECT_ROOT / "local_hom_checkpoint.pt",
    local_hom_model,
    local_hom_history,
    local_hom_elapsed,
    local_hom_prms,
    extra={"label": LOCAL_HOM_LABEL},
)

local_hom_train_loss = np.array(local_hom_history["train_loss"])
local_hom_test_loss = np.array(local_hom_history["test_loss"])
local_hom_train_acc = np.array(local_hom_history["train_acc"])
local_hom_test_acc = np.array(local_hom_history["test_acc"])

local_hom_alpha_after, local_hom_beta_after = get_hidden_ab_tensors(local_hom_model)
local_hom_after = summarize_hidden_taus(local_hom_model, local_hom_prms["time_step"])
local_hom_abs_alpha_shift = np.abs(local_hom_alpha_after.detach().cpu().numpy().ravel() - local_hom_alpha_before.detach().cpu().numpy().ravel())
local_hom_abs_beta_shift = np.abs(local_hom_beta_after.detach().cpu().numpy().ravel() - local_hom_beta_before.detach().cpu().numpy().ravel())

training_results[LOCAL_HOM_LABEL] = {
    "history": local_hom_history,
    "elapsed_s": float(local_hom_elapsed),
    "test_acc": float(local_hom_test_acc[-1]),
    "test_loss": float(local_hom_test_loss[-1]),
    "initial_summary": local_hom_before,
    "final_summary": local_hom_after,
    "mean_abs_alpha_change": float(local_hom_abs_alpha_shift.mean()),
    "mean_abs_beta_change": float(local_hom_abs_beta_shift.mean()),
    "checkpoint_path": str(local_hom_checkpoint_path),
}

training_results[LOCAL_HET_LABEL].update({
    "history": local_het_history,
    "elapsed_s": float(local_het_elapsed),
    "test_acc": float(local_het_history["test_acc"][-1]),
    "test_loss": float(local_het_history["test_loss"][-1]),
    "checkpoint_path": str(CHECKPOINT_PATH),
})

print(
    f"Local-Hom final test acc: {local_hom_test_acc[-1]:.3f} | total time: {local_hom_elapsed/60:.1f} min"
)
print(f"Saved Local-Hom checkpoint to: {local_hom_checkpoint_path}")
print(
    f"Local-Het checkpoint test acc: {local_het_history['test_acc'][-1]:.3f} | "
    f"saved time: {local_het_elapsed/60:.1f} min"
)

## Performance and Tau Evaluation

These cells adapt the post-training comparison section from the initial notebook to `Local-Hom` versus the checkpoint-loaded `Local-Het`.


In [ ]:
required_names = [
    "local_hom_history",
    "local_het_history",
    "training_results",
    "local_hom_alpha_after",
    "local_hom_beta_after",
    "local_het_alpha_after",
    "local_het_beta_after",
    "local_hom_prms",
    "local_het_prms",
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(f"Run the Local-Hom training cell first. Missing: {missing}")

epochs = np.arange(1, len(local_hom_history["train_acc"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(epochs, local_hom_history["train_acc"], marker="o", label=f"{LOCAL_HOM_LABEL} train")
axes[0].plot(epochs, local_hom_history["test_acc"], marker="o", linestyle="--", label=f"{LOCAL_HOM_LABEL} test")
axes[0].plot(epochs, local_het_history["train_acc"], marker="o", label=f"{LOCAL_HET_LABEL} train")
axes[0].plot(epochs, local_het_history["test_acc"], marker="o", linestyle="--", label=f"{LOCAL_HET_LABEL} test")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Task Performance Across Epochs")
axes[0].grid(alpha=0.2)
axes[0].legend()

final_accs = [training_results[LOCAL_HOM_LABEL]["test_acc"], training_results[LOCAL_HET_LABEL]["test_acc"]]
bars = axes[1].bar([LOCAL_HOM_LABEL, LOCAL_HET_LABEL], final_accs, color=["#4c78a8", "#f58518"], alpha=0.9)
axes[1].set_ylabel("Final test accuracy")
axes[1].set_title("Final SHD Task Performance")
axes[1].set_ylim(0, max(final_accs) * 1.25 if max(final_accs) > 0 else 1.0)
axes[1].grid(axis="y", alpha=0.2)
for bar, value in zip(bars, final_accs):
    axes[1].text(bar.get_x() + bar.get_width() / 2, value, f"{value:.3f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

local_hom_tau_syn_ms = decay_to_tau(local_hom_alpha_after.detach().cpu().numpy().ravel(), local_hom_prms["time_step"]) * 1e3
local_hom_tau_mem_ms = decay_to_tau(local_hom_beta_after.detach().cpu().numpy().ravel(), local_hom_prms["time_step"]) * 1e3
local_het_tau_syn_ms = decay_to_tau(local_het_alpha_after.detach().cpu().numpy().ravel(), local_het_prms["time_step"]) * 1e3
local_het_tau_mem_ms = decay_to_tau(local_het_beta_after.detach().cpu().numpy().ravel(), local_het_prms["time_step"]) * 1e3

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(local_het_tau_syn_ms, bins=20, color="#54a24b", alpha=0.85, edgecolor="white", label=LOCAL_HET_LABEL)
axes[0].axvline(local_hom_tau_syn_ms[0], color="#4c78a8", linestyle="--", linewidth=2, label=f"{LOCAL_HOM_LABEL} {local_hom_tau_syn_ms[0]:.1f} ms")
axes[0].set_title(f"{LOCAL_HET_LABEL} Tau Syn Distribution")
axes[0].set_xlabel("tau_syn (ms)")
axes[0].set_ylabel("Neuron count")
axes[0].grid(axis="y", alpha=0.2)
axes[0].legend()

axes[1].hist(local_het_tau_mem_ms, bins=20, color="#e45756", alpha=0.85, edgecolor="white", label=LOCAL_HET_LABEL)
axes[1].axvline(local_hom_tau_mem_ms[0], color="#4c78a8", linestyle="--", linewidth=2, label=f"{LOCAL_HOM_LABEL} {local_hom_tau_mem_ms[0]:.1f} ms")
axes[1].set_title(f"{LOCAL_HET_LABEL} Tau Mem Distribution")
axes[1].set_xlabel("tau_mem (ms)")
axes[1].set_ylabel("Neuron count")
axes[1].grid(axis="y", alpha=0.2)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
tau_distributions = {
    "run_config": {
        "local_hom_prms": {k: v for k, v in local_hom_prms.items() if isinstance(v, (int, float, str, bool, list, tuple))},
        "local_het_prms": {k: v for k, v in local_het_prms.items() if isinstance(v, (int, float, str, bool, list, tuple))},
    },
    "Local-Hom": {
        "initial": local_hom_before,
        "final": local_hom_after,
        "tau_syn_ms": local_hom_tau_syn_ms.tolist(),
        "tau_mem_ms": local_hom_tau_mem_ms.tolist(),
    },
    "Local-Het": {
        "initial": None,
        "final": local_het_after,
        "tau_syn_ms": local_het_tau_syn_ms.tolist(),
        "tau_mem_ms": local_het_tau_mem_ms.tolist(),
    },
}

save_path = PROJECT_ROOT / "Local Network Tau Distributions.json"
with open(save_path, "w") as f:
    json.dump(tau_distributions, f, indent=2)

print(f"Saved tau distributions to: {save_path}")


## Implementation A — Zero-M Null Z-Score Sweep

This is the primary runnable null-analysis implementation.

For each subset size from 2 to 32 neurons and for each network:
- compute the observed $W$- and $\mathcal{M}$-information,
- generate a junk null distribution from random Gaussian processes with true $\mathcal{M} \approx 0$,
- compare the observed values against that null distribution,
- report subset-wise z-scores and empirical upper-tail p-values.

This answers the question: *is the observed $\mathcal{M}$ larger than what the estimator would produce on random Gaussian dynamics with no structured multivariate predictive interaction?*

**Autosave / resume:** after each completed subset size, the sweep writes results to one JSON file per network in the `Project Files` folder. Re-running the sweep cell resumes from the saved subsets instead of restarting from 2.


In [ ]:
required_names = [
    "training_results",
    "local_hom_model",
    "local_het_model",
    "local_hom_prms",
    "local_het_prms",
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(f"Run the setup cell and Local-Hom training cell first. Missing: {missing}")

hidden_size = int(local_hom_model.network[0].output_size)
subset_sizes = list(range(2, hidden_size + 1))
m_downsample_stride = RUN_CONFIG.get("m_downsample_stride", 4)
m_ridge = RUN_CONFIG.get("m_ridge", 1e-2)
null_samples = RUN_CONFIG.get("null_samples", 200)
null_mode = "iid"

local_hom_zero_null_path = PROJECT_ROOT / "Project Files" / "local_hom_zero_m_null_sweep.json"
local_het_zero_null_path = PROJECT_ROOT / "Project Files" / "local_het_zero_m_null_sweep.json"

print(
    f"Running Implementation A for {LOCAL_HOM_LABEL}: zero-M Gaussian null with N={null_samples} per subset...\n"
    f"Autosave file: {local_hom_zero_null_path.name}"
)
local_hom_zero_null_sweep = sweep_zero_m_null_by_subset(
    local_hom_model,
    prms=local_hom_prms,
    path=SHD_TEST,
    subset_sizes=subset_sizes,
    max_batches=RUN_CONFIG["m_batches"],
    downsample_stride=m_downsample_stride,
    ridge=m_ridge,
    n_null=null_samples,
    null_mode=null_mode,
    seed=1000,
    save_path=local_hom_zero_null_path,
    resume=True,
    label=LOCAL_HOM_LABEL,
)

print(
    f"\nRunning Implementation A for {LOCAL_HET_LABEL}: zero-M Gaussian null with N={null_samples} per subset...\n"
    f"Autosave file: {local_het_zero_null_path.name}"
)
local_het_zero_null_sweep = sweep_zero_m_null_by_subset(
    local_het_model,
    prms=local_het_prms,
    path=SHD_TEST,
    subset_sizes=subset_sizes,
    max_batches=RUN_CONFIG["m_batches"],
    downsample_stride=m_downsample_stride,
    ridge=m_ridge,
    n_null=null_samples,
    null_mode=null_mode,
    seed=2000,
    save_path=local_het_zero_null_path,
    resume=True,
    label=LOCAL_HET_LABEL,
)

comparison_results = deepcopy(training_results)
comparison_results[LOCAL_HOM_LABEL]["zero_m_null_sweep"] = local_hom_zero_null_sweep
comparison_results[LOCAL_HOM_LABEL]["zero_m_null_path"] = str(local_hom_zero_null_path)
comparison_results[LOCAL_HET_LABEL]["zero_m_null_sweep"] = local_het_zero_null_sweep
comparison_results[LOCAL_HET_LABEL]["zero_m_null_path"] = str(local_het_zero_null_path)

comparison_results[LOCAL_HOM_LABEL]["M_meta"] = largest_valid_sweep_result(local_hom_zero_null_sweep, key_prefix="zero_null")
comparison_results[LOCAL_HET_LABEL]["M_meta"] = largest_valid_sweep_result(local_het_zero_null_sweep, key_prefix="zero_null")
comparison_results[LOCAL_HOM_LABEL]["M_bits"] = comparison_results[LOCAL_HOM_LABEL]["M_meta"]["observed_M_bits"]
comparison_results[LOCAL_HOM_LABEL]["W_bits"] = comparison_results[LOCAL_HOM_LABEL]["M_meta"]["observed_W_bits"]
comparison_results[LOCAL_HET_LABEL]["M_bits"] = comparison_results[LOCAL_HET_LABEL]["M_meta"]["observed_M_bits"]
comparison_results[LOCAL_HET_LABEL]["W_bits"] = comparison_results[LOCAL_HET_LABEL]["M_meta"]["observed_W_bits"]


def _subset_result(results, subset_size):
    for row in results:
        if row["subset_size"] == subset_size:
            return row
    return None


for label, sweep in [(LOCAL_HOM_LABEL, local_hom_zero_null_sweep), (LOCAL_HET_LABEL, local_het_zero_null_sweep)]:
    full_result = _subset_result(sweep, hidden_size)
    largest_valid = largest_valid_sweep_result(sweep, key_prefix="zero_null")
    print(f"\n{label} implementation A summary")
    print("-" * 60)
    if full_result is not None and np.isfinite(full_result["M_zscore"]):
        print(
            f"subset=32  observed M={full_result['observed_M_bits']:.4f}  "
            f"null mean={full_result['null_M_mean']:.4f}  "
            f"z={full_result['M_zscore']:.2f}  p={full_result['M_p_upper']:.4f}"
        )
    elif full_result is not None:
        print(f"subset=32  failed  {full_result.get('error', 'unknown error')}")
    print(
        f"largest valid subset={largest_valid['subset_size']:02d}  "
        f"observed M={largest_valid['observed_M_bits']:.4f}  "
        f"z={largest_valid['M_zscore']:.2f}  p={largest_valid['M_p_upper']:.4f}"
    )


## Implementation A Plots — Observed M/W and Zero-M Null Z-Scores


In [ ]:
required_names = ["comparison_results", "local_hom_zero_null_sweep", "local_het_zero_null_sweep"]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(f"Run Implementation A first. Missing: {missing}")


def _series_from_sweep(results, key):
    x = np.array([row["subset_size"] for row in results], dtype=int)
    y = np.array([row[key] for row in results], dtype=float)
    return x, y


hom_x, hom_m = _series_from_sweep(local_hom_zero_null_sweep, "observed_M_bits")
het_x, het_m = _series_from_sweep(local_het_zero_null_sweep, "observed_M_bits")
hom_x_w, hom_w = _series_from_sweep(local_hom_zero_null_sweep, "observed_W_bits")
het_x_w, het_w = _series_from_sweep(local_het_zero_null_sweep, "observed_W_bits")
hom_x_z, hom_m_z = _series_from_sweep(local_hom_zero_null_sweep, "M_zscore")
het_x_z, het_m_z = _series_from_sweep(local_het_zero_null_sweep, "M_zscore")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(hom_x, hom_m, marker="o", color="#4c78a8", label=LOCAL_HOM_LABEL)
axes[0].plot(het_x, het_m, marker="o", color="#f58518", label=LOCAL_HET_LABEL)
axes[0].set_xlabel("Subset size")
axes[0].set_ylabel("Observed M-info (bits)")
axes[0].set_title("Observed M-info by Subset Size")
axes[0].grid(alpha=0.2)
axes[0].legend()

axes[1].plot(hom_x_w, hom_w, marker="o", color="#72b7b2", label=LOCAL_HOM_LABEL)
axes[1].plot(het_x_w, het_w, marker="o", color="#eeca3b", label=LOCAL_HET_LABEL)
axes[1].set_xlabel("Subset size")
axes[1].set_ylabel("Observed W-info (bits)")
axes[1].set_title("Observed W-info by Subset Size")
axes[1].grid(alpha=0.2)
axes[1].legend()

axes[2].plot(hom_x_z, hom_m_z, marker="o", color="#4c78a8", label=LOCAL_HOM_LABEL)
axes[2].plot(het_x_z, het_m_z, marker="o", color="#f58518", label=LOCAL_HET_LABEL)
axes[2].axhline(0.0, color="black", linewidth=1, alpha=0.6)
axes[2].axhline(1.96, color="#888888", linestyle="--", linewidth=1, alpha=0.8, label="z=1.96")
axes[2].set_xlabel("Subset size")
axes[2].set_ylabel("M-info z-score vs zero-M null")
axes[2].set_title("Implementation A: Zero-M Null Z-Scores")
axes[2].grid(alpha=0.2)
axes[2].legend()

plt.tight_layout()
plt.show()

full_subset = int(local_hom_prms["nb_recurrent"])
full_hom = next((row for row in local_hom_zero_null_sweep if row["subset_size"] == full_subset), None)
full_het = next((row for row in local_het_zero_null_sweep if row["subset_size"] == full_subset), None)

print("Implementation A final subset summary")
print("-" * 72)
for label, row in [(LOCAL_HOM_LABEL, full_hom), (LOCAL_HET_LABEL, full_het)]:
    if row is not None and np.isfinite(row["M_zscore"]):
        print(
            f"{label:10s} subset=32  obsM={row['observed_M_bits']:.4f}  "
            f"nullM={row['null_M_mean']:.4f}  z={row['M_zscore']:.2f}  p={row['M_p_upper']:.4f}"
        )
    elif row is not None:
        print(f"{label:10s} subset=32  FAILED  {row.get('error', 'unknown error')}")
    else:
        print(f"{label:10s} subset=32  missing")


## Implementation B — Deferred Matched-Information Null (Design Only)

This section is intentionally markdown-only for now.

### Goal
Construct a stricter null model following the thesis `Null Info.pdf` method: compare the observed $\mathcal{M}$-information against an ensemble of random Gaussian systems with the **same total mutual information** as the observed subset, rather than against a zero-$\mathcal{M}$ junk distribution.

### Why this is stricter
Implementation A asks whether the observed $\mathcal{M}$ is larger than what appears under random Gaussian dynamics with no structured multivariate predictive interaction. That is useful for checking estimator bias and basic significance.

Implementation B would go further by controlling for the overall predictive information budget. It asks whether the observed $\mathcal{M}$ is unusual **even among random Gaussian systems that have the same total mutual information**.

### Thesis-based construction
For each subset size $k$ and each network:
1. Compute the observed lagged covariance and the observed TDMI / TMI.
2. Sample random Gaussian null systems by drawing:
   - a random coefficient matrix $\tilde{A}$,
   - a random source covariance $\tilde{\Sigma}_S$,
   - a random noise covariance $\tilde{\Sigma}_{\epsilon}$.
3. Solve for the scalar noise parameter $g$ so that the null system satisfies the same total mutual information as the observed subset.
4. Build the null covariance matrix.
5. Compute $W$ and $\mathcal{M}$ for that null covariance.
6. Repeat many times to obtain null distributions of $W$ and $\mathcal{M}$.
7. Compare the observed values against that matched-information null via z-scores, empirical p-values, and quantiles.

### Practical plan for later conversion to Python
- Use the observed subset dimensionality as the null dimensionality.
- Use one deterministic subset per size for the first pass, matching Implementation A.
- Sample null systems directly in covariance form to avoid simulating long time series.
- Solve for $g$ with a robust scalar root finder such as bisection or Brent.
- Cache partial results after each subset size so long null runs can resume.
- Report both the full-size result at $k=32$ and the largest valid subset if numerical failures occur.

### Interpretation
- Implementation A: estimator-floor / zero-$\mathcal{M}$ baseline.
- Implementation B: matched-information significance test aligned with the thesis null-model logic.

If this section is later converted to code, it should live after Implementation A and use the same subset-size sweep structure so the two null analyses remain directly comparable.
